# بازشناسی احساس گفتار با صوت و متن روی ShEMO — نسخهٔ پروژه‌ای کد مقاله

این **تنها نوت‌بوک** پروژه است. همهٔ منطق در پکیج پایتونی `atser/` است؛ این نوت‌بوک فقط آن را import می‌کند،
مراحل را اجرا می‌کند و نتیجه‌ها را نشان می‌دهد.

**مقاله:** *Audio-Textual Emotion Recognition Using Pre-trained Models* (Dehghani et al., 2025) —
[کد اصلی نویسنده](https://github.com/ZahraDehghani99/Audio-Textual-Emotion-Recognition-using-Pre-trained-models)

### ساختار پروژه

| فایل | کار |
| --- | --- |
| `atser/config.py` | همهٔ تنظیمات به شکل dataclass. `paper_shemo()` تنظیمات دقیق نوت‌بوک‌های نویسنده است |
| `atser/data/shemo.py` | خواندن CSV نشست‌های نویسنده، پیدا کردن فایل‌های wav، تبدیل به ۱۶ kHz |
| `atser/data/splits.py` | تقسیم train/valid/test هر fold با همان فراخوانی‌های `datasets` نویسنده |
| `atser/models/finetune.py` | fine-tune مدل صوتی (wav2vec2) و متنی (BERT)، fold به fold، با قابلیت ادامه |
| `atser/features/extract.py` | بردار هر جمله: «میانگینِ mean-pooling همهٔ لایه‌ها» (روش مقاله) + بردار هر لایه |
| `atser/fusion/` | روش‌های ترکیب: روش مقاله و ۱۲ روش دیگر. روش جدید = یک کلاس |
| `atser/evaluation/` | معیارهای مقاله، آزمون آماری، جدول‌ها و نمودارها |
| `atser/pipeline/` | ذخیره و بازیابی خروجی هر مرحله و اجرای مرحله‌به‌مرحله |
| `atser/persist.py` | ذخیرهٔ ماندگار: Kaggle Dataset، Google Drive، Hugging Face |
| `tests/check_against_author.py` | اجرای کد خود نویسنده و این پکیج کنار هم و مقایسهٔ خروجی‌ها |

### آیا این بازنویسی دقیقاً همان کد نویسنده است؟

`tests/check_against_author.py` سلول‌های نوت‌بوک‌های نویسنده را (فقط با وصله‌های Colab/Hub) و این پکیج را روی
یک دادهٔ یکسان اجرا می‌کند. روی CPU و با مدل‌های کوچک، که نتیجه قطعی و تکرارپذیر است:

| مرحله | نتیجه |
| --- | --- |
| fine-tune صوت، fold 1 و 2 | logit های تست یکسان (بیشترین اختلاف 0.0) |
| fine-tune متن، fold 1 و 2 | یکسان (0.0) |
| ویژگی‌های صوت و متن | یکسان (0.0) |
| ترکیب «جمع + SVM» | پیش‌بینی‌ها یکسان |
| معیارهای هر کلاس (Precision تا MCC) | یکسان |

روی GPU، حتی خود کد نویسنده هم در دو اجرا بیت‌به‌بیت یکسان نمی‌شود (دو اجرای قبلی ما: صوت 75.19٪ و 76.96٪).
پس روی Kaggle انتظار «نزدیک» داریم. بخش ۴ این را خودکار بررسی می‌کند.

**یک تفاوت عمدی:** نویسنده seed را فقط یک بار اول نوت‌بوک می‌گذاشت و ۵ fold را پشت سر هم اجرا می‌کرد؛ پس
مقداردهی اولیهٔ لایهٔ آخرِ fold های ۲ تا ۵ به آموزش fold قبلی بستگی داشت. اینجا seed هر fold جدا گذاشته
می‌شود تا اجرای نیمه‌کاره، بعداً با همان نتیجه ادامه پیدا کند.

### مراحل و زمان تقریبی روی T4 (از اجرای قبلی شما)

| مرحله | زمان |
| --- | --- |
| fine-tune صوت، ۵ fold (یک بار برای همهٔ آزمایش‌ها) | حدود ۱۵۰ دقیقه |
| fine-tune متن، ۵ fold (برای هر مدل متنی) | ۱۰ تا ۲۵ دقیقه |
| استخراج ویژگی صوت / متن | حدود ۲۰ / ۳ دقیقه |
| همهٔ روش‌های ترکیب | چند دقیقه |

هر مرحله خروجی‌اش را در `/kaggle/working/atser_store` ذخیره می‌کند. اگر session قطع شد، اجرای دوباره از
همان‌جا ادامه می‌دهد.

### اجرا روی Kaggle

1. **Settings:** Accelerator = GPU T4 x2 (فقط یکی استفاده می‌شود، مثل نویسنده)، Internet = On.
2. **Add Input:** دیتاست `mansourehk/shemo-persian-speech-emotion-detection-database`.
3. **اختیاری:** خروجی اجرای قبلی همین نوت‌بوک (Add Input → Your Work) تا مدل‌های آموزش‌دیده دوباره استفاده شوند.
4. بخش ۰ را تنظیم کنید و **Run All**.
5. برای نگه داشتن خروجی: **Save Version → Save & Run All**. (Quick Save خروجی را نگه نمی‌دارد.)

## ۰. تنظیمات

فقط همین سلول را تغییر دهید.

| تنظیم | معنی |
| --- | --- |
| `QUICK_TEST` | `True`: فقط ۲۰ جمله از هر کلاس در هر fold و ۱ epoch. حدود ۱۰ دقیقه، برای اطمینان از اینکه همه‌چیز اجرا می‌شود. نتایجش معنی ندارد |
| `RUN_REPRODUCTION` | گام ۱: خود مقاله (wav2vec2-base + bert-base-uncased + جمع + SVM) |
| `TEXT_MODELS` | گام ۲: مدل‌های متنی دیگر. مدل صوتی دوباره آموزش نمی‌بیند |
| `FUSION_METHODS` | گام ۳: روش‌های ترکیب که روی همهٔ آزمایش‌ها مقایسه می‌شوند (فهرست کامل در بخش ۱ چاپ می‌شود) |
| `FOLD_SCHEME` | `"paper"`: fold های نویسنده (گویندهٔ تست در آموزش هم هست). `"speaker"`: مستقل از گوینده |
| `FEATURE_SOURCE` | `"finetuned"`: ویژگی از مدل fine-tune شدهٔ هر fold (مقاله). `"pretrained"`: از مدل اصلی، بدون آموزش |
| `EXPORT_*` | ذخیرهٔ ماندگار در Kaggle Dataset (بخش ۷) |

In [ ]:
# ---------------- چه چیزی اجرا شود ----------------
QUICK_TEST = False
QUICK_ROWS = 20                    # در حالت QUICK_TEST: تعداد جمله از هر کلاس در هر fold

RUN_REPRODUCTION = True            # گام ۱: خود مقاله
TEXT_MODELS = [                    # گام ۲: مدل‌های متنی دیگر (نام مدل روی Hugging Face)
    "HooshvareLab/bert-fa-base-uncased",
]
FUSION_METHODS = [                 # گام ۳: (نام روش، پارامترها)
    ("paper_sum_svm", {}),         # روش مقاله
    ("concat_svm", {}),
    ("scaled_concat_svm", {}),
    ("concat_logreg", {}),
    ("stacking_oof", {}),
    ("late_average", {"w_audio": 0.5}),
    ("late_product", {}),
    ("late_confidence", {}),
    ("mlp_concat", {}),
    ("gmu", {}),
    ("weighted_layers_mlp", {}),
    ("audio_svm", {}),             # مرجع: فقط صوت، روی همان ویژگی‌ها
    ("text_svm", {}),              # مرجع: فقط متن، روی همان ویژگی‌ها
]
FOLD_SCHEME = "paper"              # "paper" یا "speaker"
FEATURE_SOURCE = "finetuned"       # "finetuned" یا "pretrained"

# ---------------- کد پروژه ----------------
CODE_SOURCE = "github"             # "github" یا مسیر یک پوشهٔ محلی که atser/ داخلش است
CODE_REPO = "https://github.com/mo0o0o0os/Persian-GANwriting.git"
CODE_BRANCH = "claude/hopeful-gauss-7sojs1"
CODE_SUBDIR = "audio_textual_ser"

# ---------------- ذخیره ----------------
RESTORE_PREVIOUS = True            # استفاده از خروجی اجراهای قبلی که به نوت‌بوک اضافه شده‌اند
MAKE_RESULTS_ZIP = True            # یک zip از جدول‌ها، نمودارها و پیش‌بینی‌ها (برای دانلود)
EXPORT_TO_KAGGLE_DATASET = False   # نیاز به secret های KAGGLE_USERNAME و KAGGLE_KEY
KAGGLE_DATASET_SLUG = "atser-shemo-store"
EXPORT_INCLUDE_MODELS = False      # مدل‌ها چند گیگابایت‌اند؛ بدون آن‌ها ویژگی‌ها و پیش‌بینی‌ها ذخیره می‌شوند
SAVE_TO_DRIVE = False              # فقط Colab

# ---------------- پیشرفته (معمولاً دست نزنید) ----------------
EXTRA_DATA_DIRS = []               # پوشه‌های دیگری که فایل‌های wav ShEMO در آن‌ها هست
MODEL_OVERRIDES = {}               # نام مدل -> پوشهٔ محلی (برای اجرای آفلاین/آزمایشی)

## ۱. آماده‌سازی محیط و دریافت کد

- قبل از import شدن torch فقط یک GPU دیده می‌شود (`CUDA_VISIBLE_DEVICES=0`)، مثل Colab نویسنده. با دو GPU،
  Trainer خودش DataParallel می‌زند و batch مؤثر و نتیجه عوض می‌شود.
- کد پروژه از GitHub (شاخهٔ بالا) در `/tmp/atser_code` دریافت می‌شود. در خروجی ذخیره نمی‌شود، پس همیشه آخرین نسخه است.
- فقط پکیج‌هایی نصب می‌شوند که نیستند. نصب دوبارهٔ `librosa` یا `numpy` روی Kaggle ممکن است محیط را خراب کند.

In [ ]:
import os, sys, subprocess, importlib.util
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # one GPU, like the author (must run before torch is imported)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

if CODE_SOURCE == "github":
    code_root = Path("/tmp/atser_code")
    if (code_root / ".git").exists():
        subprocess.check_call(["git", "-C", str(code_root), "fetch", "-q", "--depth", "1", "origin", CODE_BRANCH])
        subprocess.check_call(["git", "-C", str(code_root), "reset", "-q", "--hard", "FETCH_HEAD"])
    else:
        subprocess.check_call(["git", "clone", "-q", "--depth", "1", "-b", CODE_BRANCH, CODE_REPO, str(code_root)])
    CODE_DIR = code_root / CODE_SUBDIR
    commit = subprocess.check_output(["git", "-C", str(code_root), "rev-parse", "--short", "HEAD"], text=True).strip()
else:
    CODE_DIR, commit = Path(CODE_SOURCE), "local"
sys.path.insert(0, str(CODE_DIR))

missing = [p for p in ("soundfile", "librosa", "openpyxl") if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import pandas as pd
from IPython.display import display, Markdown
import atser
from atser import env, fusion, persist
from atser.config import MODEL_OVERRIDES as _OVR, DataConfig, FusionSpec, paper_shemo
from atser.data import ShEMO, download_shemo_with_kagglehub, fetch_author_repo
from atser.evaluation import plots, report
from atser.pipeline import ArtifactStore, Experiment, find_stores
from atser.utils import log

_OVR.update(MODEL_OVERRIDES)
pd.set_option("display.max_colwidth", 120)
log(f"atser {atser.__version__} از {CODE_DIR} (commit {commit})")
display(pd.Series(env.describe_hardware(), name="محیط").to_frame())
display(Markdown("**روش‌های ترکیب موجود** (`atser.fusion.available()`):"))
display(fusion.available())

## ۲. داده

- **fold ها:** همان ۵ فایل `ShEMO/Dataset/Sessions/session_*.csv` ریپوی نویسنده (در commit بازتولیدشده).
  متن‌ها نسخهٔ اصلاح‌شدهٔ [modified-shemo](https://github.com/aliyzd95/modified-shemo) هستند.
- **صدا:** فایل‌های اصلی ۴۴.۱ kHz از دیتاست Kaggle، دقیقاً مثل `datasets.Audio(sampling_rate=16000)` نویسنده
  (float64 و `librosa.resample`) یک بار به ۱۶ kHz تبدیل و در `/tmp/atser_cache` نگه داشته می‌شوند.
- **جدول زیر:** تعداد جمله‌های هر کلاس در هر fold، تعداد گوینده‌ها، و درصد جمله‌های تستی که گوینده‌شان در
  داده‌های آموزش هم هست. در fold های نویسنده این عدد تقریباً ۱۰۰٪ است، یعنی ارزیابی **وابسته به گوینده** است.

In [ ]:
paths = env.default_paths()
author_repo = fetch_author_repo(paths.cache / "author_repo")
data_cfg = DataConfig(fold_scheme=FOLD_SCHEME, quick_rows_per_class=QUICK_ROWS if QUICK_TEST else 0)
wav_roots = [*paths.inputs, *map(Path, EXTRA_DATA_DIRS)]
try:
    data = ShEMO.prepare(author_repo / "ShEMO/Dataset/Sessions", wav_roots, paths.cache, data_cfg)
except FileNotFoundError as err:
    log(f"{err}\nتلاش برای دانلود با kagglehub ...")
    data = ShEMO.prepare(author_repo / "ShEMO/Dataset/Sessions", [*wav_roots, download_shemo_with_kagglehub()],
                         paths.cache, data_cfg)
display(data.summary(data_cfg))
display(data.table[["uid", "utterance_name", "label", "session", "speaker", "seconds", "text"]].head())

## ۳. استفاده از نتایج اجراهای قبلی

همه‌چیز در `atser_store` ذخیره می‌شود (مدل‌ها، ویژگی‌ها، پیش‌بینی‌ها). اگر خروجی یک اجرای قبلی را با
**Add Input → Your Work** یا به شکل Dataset به نوت‌بوک اضافه کرده باشید، این سلول هر چیزی را که اینجا نیست
کپی می‌کند؛ مرحله‌هایی که کامل شده‌اند دوباره اجرا نمی‌شوند. آموزش نیمه‌کاره هم از آخرین epoch ادامه پیدا می‌کند.

In [ ]:
store = ArtifactStore(paths.store)
if RESTORE_PREVIOUS:
    previous = find_stores(paths.inputs, exclude=store.root)
    log(f"{len(previous)} خروجی قبلی پیدا شد: {[str(p) for p in previous]}")
    store.restore_from(previous)
display(store.usage())

## ۴. گام ۱ — بازتولید دقیق مقاله

تنظیمات `paper_shemo()` کلمه‌به‌کلمه از نوت‌بوک‌های نویسنده برداشته شده:

| | صوت | متن |
| --- | --- | --- |
| مدل | `facebook/wav2vec2-base` | `bert-base-uncased` |
| epoch / lr / weight decay | 6 / 1e-4 / 2e-5 | 4 / 5e-5 / 0.1 |
| batch | 16 × grad-accum 2 | 32 |
| warmup | 10٪ | 10٪ |
| ورودی | حداکثر ۵ ثانیه، CNN منجمد | truncation به ۵۱۲ توکن |
| انتخاب مدل | بهترین epoch روی valid (accuracy) | آخرین epoch |
| seed / valid | 1968 / ۱۰٪ هر fold | 42 / ۱۰٪ هر fold (طبقه‌بندی‌شده) |

سپس ویژگی‌ها از مدل fine-tune شدهٔ هر fold استخراج و همهٔ `FUSION_METHODS` اجرا می‌شوند.
`exp.status()` نشان می‌دهد کدام مرحله‌ها قبلاً انجام شده‌اند (✓).

In [ ]:
specs = [FusionSpec.of(method, **params) for method, params in FUSION_METHODS]


def make_config(text_model):
    cfg = paper_shemo(text_model, FOLD_SCHEME).with_features(source=FEATURE_SOURCE).with_fusion(*specs)
    return cfg.quick(QUICK_ROWS) if QUICK_TEST else cfg


experiments = {}
if RUN_REPRODUCTION:
    base_cfg = make_config("bert-base-uncased")
    exp = Experiment(base_cfg, data, store)
    experiments[base_cfg.name] = exp
    display(exp.status())
    exp.run()
    display(exp.status())

### بررسی بازتولید

مقایسهٔ UA این اجرا با سه مرجع: جدول مقاله، خروجی‌های ذخیره‌شده در ریپوی نویسنده، و اجرای قبلی ما با
نوت‌بوک‌های خود ریپو روی Kaggle. «بازتولید شد» یعنی اختلاف میانگین با همهٔ مرجع‌ها حداکثر ۲.۵ واحد درصد است.
ستون‌های `p vs ...` آزمون t جفتی روی ۵ fold است. p بزرگ یعنی تفاوتی که ببینیم از نوسان عادی بیشتر نیست.

In [ ]:
if RUN_REPRODUCTION:
    df_repro = report.collect(store, [base_cfg.name])
    if FOLD_SCHEME == "paper" and FEATURE_SOURCE == "finetuned" and not QUICK_TEST:
        display(report.reproduction_check(df_repro, base_cfg.name))
    else:
        log("مقایسه با مقاله فقط برای fold های نویسنده، ویژگی finetuned و اجرای کامل معنی دارد.")
    display(report.per_fold(df_repro[df_repro["system"].isin(["audio", "text", "paper_sum_svm"])]))

## ۵. گام ۲ — مدل‌های متنی دیگر

برای هر مدل در `TEXT_MODELS` فقط مدل متنی آموزش می‌بیند. تنظیمات صوت همان است، پس مدل‌ها و ویژگی‌های صوتی
از همان پوشه خوانده می‌شوند. تنظیمات آموزش متن هم همان تنظیمات نویسنده است، فقط نام مدل عوض می‌شود.

In [ ]:
for text_model in TEXT_MODELS:
    cfg = make_config(text_model)
    exp = Experiment(cfg, data, store)
    experiments[cfg.name] = exp
    exp.run()
    display(exp.status())

## ۶. گام ۳ — مقایسهٔ همه‌چیز

همهٔ جدول‌ها از پیش‌بینی‌های ذخیره‌شده ساخته می‌شوند. در `atser_results/` هم به شکل CSV و Excel ذخیره می‌شوند.

| جدول / نمودار | چه نشان می‌دهد |
| --- | --- |
| خلاصه | میانگین ± انحراف معیار ۵ fold برای UA، WA، F1، W-F1، MCC |
| رتبه‌بندی | همهٔ سیستم‌ها مرتب بر اساس UA |
| هر fold | UA هر fold. نوسان بین fold ها را نشان می‌دهد |
| مقایسه با صوت | هر سیستم در برابر صوت تنها: اختلاف، آزمون t جفتی، Wilcoxon، تصحیح Holm |
| regret | بهترینِ صوت و متن منهای ترکیب. مثبت یعنی ترکیب ضرر زده |
| هر کلاس | recall هر کلاس برای هر سیستم |
| overfit | UA روی دادهٔ آموزش در برابر تست. فاصلهٔ زیاد یعنی حفظ کردن |
| منحنی آموزش | loss و accuracy اعتبارسنجی در هر epoch، برای هر fold |

In [ ]:
RESULTS = paths.results
RESULTS.mkdir(parents=True, exist_ok=True)
df = report.collect(store, list(experiments))
tables = {}

tables["summary"] = report.summary(df)
display(Markdown("### خلاصه (٪، میانگین ± انحراف معیار روی fold ها)"))
display(tables["summary"])

display(Markdown("### رتبه‌بندی همهٔ سیستم‌ها"))
display(report.leaderboard(df))

tables["per_fold_UA"] = report.per_fold(df)
display(Markdown("### UA هر fold"))
display(tables["per_fold_UA"])

In [ ]:
tables["vs_audio"] = report.compare_to(df, baseline="audio")
display(Markdown("### هر سیستم در برابر صوت تنها (آزمون جفتی روی fold ها)"))
display(tables["vs_audio"])

tables["regret"] = report.regret(df)
display(Markdown("### regret: بهترین تک‌وجهی منهای ترکیب (واحد درصد)"))
display(tables["regret"])

In [ ]:
tables["per_class_recall"] = report.per_class(store, list(experiments))
display(Markdown("### recall هر کلاس (٪)"))
display(tables["per_class_recall"])
plots.heatmap(tables["per_class_recall"], "Per-class recall (%)", save_dir=RESULTS, name="per_class_recall")
plots.score_bars(df, "UA", references={"paper audio 74.83": 74.83, "paper fusion 76.37": 76.37}, save_dir=RESULTS)
# نمودار هر fold فقط برای صوت، متن، روش مقاله و دو روش برتر هر آزمایش (برای خوانایی)
plots.per_fold_lines(report.key_systems(df, top=2), "UA", save_dir=RESULTS);

In [ ]:
# ماتریس درهم‌ریختگی (نرمال‌شده، همهٔ fold ها با هم): صوت، متن، روش مقاله و بهترین روش ترکیب هر آزمایش
matrices = {}
for name in experiments:
    part = df[df["experiment"] == name]
    fusion_means = part[part["kind"] == "fusion"].groupby("system")["UA"].mean()
    wanted = ["audio", "text", "paper_sum_svm"]
    if len(fusion_means):
        wanted.append(fusion_means.idxmax())
    for system in dict.fromkeys(wanted):
        if system in set(part["system"]):
            matrices[f"{name}\n{system}"] = report.confusion(store, name, system, data.classes)
plots.confusion_grid(matrices, data.classes, save_dir=RESULTS);

In [ ]:
tables["overfitting"] = report.overfitting(store, df)
display(Markdown("### overfit: UA آموزش در برابر تست"))
display(tables["overfitting"])
plots.train_test_gap(tables["overfitting"], save_dir=RESULTS)
curves = report.training_curves(store)
if not curves.empty:
    plots.training_curves(curves, save_dir=RESULTS)

with pd.ExcelWriter(RESULTS / "results.xlsx") as xls:
    for key, table in tables.items():
        table.to_csv(RESULTS / f"{key}.csv")
        table.to_excel(xls, sheet_name=key[:31])
df.to_csv(RESULTS / "all_folds_all_metrics.csv", index=False)
log(f"جدول‌ها و نمودارها در {RESULTS}")

## ۷. ذخیرهٔ ماندگار

`/kaggle/working` بعد از بستن session پاک می‌شود. سه راه برای نگه داشتن خروجی:

1. **ساده‌ترین:** بعد از اجرا **Save Version → Save & Run All (Commit)**. کل `/kaggle/working` (شامل
   `atser_store` با مدل‌ها) خروجی آن نسخه می‌شود. دفعهٔ بعد: Add Input → Your Work → همین نوت‌بوک. بخش ۳
   خودش آن را پیدا می‌کند. (Quick Save خروجی را نگه نمی‌دارد.)
2. **Kaggle Dataset** (`EXPORT_TO_KAGGLE_DATASET = True`): یک Dataset خصوصی که با هر اجرا نسخهٔ جدید می‌گیرد.
   لازم است: kaggle.com → Settings → API → Create New Token، سپس در نوت‌بوک Add-ons → Secrets دو secret با
   نام‌های `KAGGLE_USERNAME` و `KAGGLE_KEY`.
3. **Colab:** `SAVE_TO_DRIVE = True` همه‌چیز را در Google Drive کپی می‌کند.

`MAKE_RESULTS_ZIP` هم یک zip کوچک (جدول‌ها، نمودارها، پیش‌بینی‌ها، تنظیمات) در Output می‌سازد که می‌شود دانلود کرد.

In [ ]:
if MAKE_RESULTS_ZIP:
    staging = persist.bundle(store, paths.exports / "results_bundle", parts=("experiments", "predictions"), include_models=False)
    import shutil
    shutil.copytree(RESULTS, staging / "atser_results", dirs_exist_ok=True)
    persist.zip_folder(staging, paths.work / "atser_results.zip")
    shutil.rmtree(staging)

if EXPORT_TO_KAGGLE_DATASET:
    staging = persist.bundle(store, paths.exports / "kaggle_dataset", include_models=EXPORT_INCLUDE_MODELS)
    persist.save_to_kaggle_dataset(staging, KAGGLE_DATASET_SLUG, title="atser ShEMO store")

if SAVE_TO_DRIVE:
    persist.save_to_drive(store.root)
display(store.usage())

## ۸. توسعه: مدل متنی، روش ترکیب یا معماری جدید

ساختار طوری است که تغییرها فقط در یک جا لازم باشد:

| می‌خواهید | کجا | ساختار بقیه |
| --- | --- | --- |
| مدل متنی دیگر | نامش را به `TEXT_MODELS` اضافه کنید | بدون تغییر؛ صوت دوباره آموزش نمی‌بیند |
| مدل صوتی دیگر (مثلاً WavLM) | `paper_shemo(...).with_audio_model("microsoft/wavlm-base-plus")` | بدون تغییر |
| تنظیمات آموزش دیگر | `with_text_model(name, epochs=2, select_best=True)` | نام آزمایش خودکار عوض می‌شود تا نتایج قاطی نشوند |
| ارزیابی مستقل از گوینده | `FOLD_SCHEME = "speaker"` | بدون تغییر |
| ویژگی بدون نشت (مدل منجمد) | `FEATURE_SOURCE = "pretrained"` | بدون تغییر |
| روش ترکیب جدید | یک کلاس با `@fusion.register("نام")` (مثال زیر) | بدون تغییر |
| سر عصبی جدید | زیرکلاس `fusion.TorchFusion` با دو تابع `inputs` و `build` | حلقهٔ آموزش آماده است |
| معماری fine-tune متفاوت | زیرکلاس `atser.models.Finetuner` با `load_processor`، `build_dataset`، `load_model` | حلقهٔ آموزش، ادامه و ذخیره آماده است |

مثال زیر یک روش ترکیب جدید تعریف و اجرا می‌کند. `RUN_EXTENSION_EXAMPLE = True` بگذارید تا اجرا شود.

In [ ]:
RUN_EXTENSION_EXAMPLE = False

import numpy as np
from atser.fusion import FusionMethod, register, softmax


@register("max_confidence")
class MaxConfidence(FusionMethod):
    # برای هر جمله، پیش‌بینی مدلی که مطمئن‌تر است
    description = "انتخاب پیش‌بینی مدلی که برای آن جمله مطمئن‌تر است"
    needs = ("logits",)

    def fit(self, train):
        return self

    def predict_proba(self, test):
        pa, pt = softmax(test.logits["audio"]), softmax(test.logits["text"])
        return np.where((pa.max(1) >= pt.max(1))[:, None], pa, pt)


if RUN_EXTENSION_EXAMPLE:
    for exp in experiments.values():
        display(exp.fuse([FusionSpec.of("max_confidence")]))